# Clean Up the AnthroFold Endpoint

When you're done running predictions, use this notebook to tear everything down — the endpoint, its endpoint config, and the underlying model. SageMaker bills you while the endpoint is `InService`, so don't forget this step.

This is the final notebook in the workflow.

## Install dependencies

In [ ]:
%pip install -qU -r requirements.txt

## Configure

The endpoint name is read from `endpoint_name.txt` (written by the deploy notebook), so you usually don't need to change anything here. To target a different endpoint, set `ANTHROFOLD_ENDPOINT_NAME` or edit `ENDPOINT_NAME` directly.

In [ ]:
import os
import time
from pathlib import Path

import boto3

SAGEMAKER_REGION = "us-east-1"

# Endpoint name is read from the sidecar file written by 1-deploy-endpoint.ipynb.
# Override by setting ANTHROFOLD_ENDPOINT_NAME or pasting the name into ENDPOINT_NAME.
ENDPOINT_NAME = os.environ.get("ANTHROFOLD_ENDPOINT_NAME", "").strip()
if not ENDPOINT_NAME:
    _sidecar = Path("endpoint_name.txt")
    if _sidecar.exists():
        ENDPOINT_NAME = _sidecar.read_text().strip()

if not ENDPOINT_NAME or ENDPOINT_NAME == "PASTE_ENDPOINT_NAME_HERE":
    raise ValueError(
        "Endpoint name not set. Set ANTHROFOLD_ENDPOINT_NAME, or paste the name "
        "into ENDPOINT_NAME above."
    )

sm = boto3.client("sagemaker", region_name=SAGEMAKER_REGION)
print(f"Endpoint: {ENDPOINT_NAME}")

## Inspect attached resources

Before we delete anything, record the endpoint config and model attached to this endpoint. That way the cleanup step only touches the resources that this endpoint actually created, leaving anything else in your account untouched.

In [ ]:
endpoint_config_name = None
model_name = None

try:
    endpoint_desc = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
    endpoint_config_name = endpoint_desc.get("EndpointConfigName")
    print(f"Endpoint status: {endpoint_desc.get('EndpointStatus')}")
except Exception as exc:
    # Endpoint already gone (e.g. a re-run). The deploy notebook names the
    # endpoint config the same as the endpoint, so still try to clean up an
    # orphaned config/model.
    print(f"Endpoint not found ({exc}); will still try to clean up its config/model.")
    endpoint_config_name = ENDPOINT_NAME

print(f"Endpoint config: {endpoint_config_name}")
if endpoint_config_name:
    try:
        config_desc = sm.describe_endpoint_config(EndpointConfigName=endpoint_config_name)
        variants = config_desc.get("ProductionVariants") or []
        model_name = variants[0].get("ModelName") if variants else None
        print(f"Model: {model_name}")
    except Exception as exc:
        print(f"Endpoint config not found ({exc}).")
        endpoint_config_name = None

## Delete

Resources come down in this order: endpoint → wait for deletion → endpoint config → model. If the endpoint is still being created (it can take ~1 hour), SageMaker may reject the delete; in that case, give it a few minutes and rerun this cell.

In [ ]:
try:
    sm.delete_endpoint(EndpointName=ENDPOINT_NAME)
    print(f"Deleted endpoint: {ENDPOINT_NAME}")
except Exception as exc:
    print(f"Endpoint delete skipped or failed: {exc}")

for _ in range(120):
    try:
        sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
        print("Endpoint still deleting; waiting 30s")
        time.sleep(30)
    except Exception:
        print("Endpoint is gone")
        break

if endpoint_config_name:
    try:
        sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
        print(f"Deleted endpoint config: {endpoint_config_name}")
    except Exception as exc:
        print(f"Endpoint config delete skipped or failed: {exc}")

if model_name:
    try:
        sm.delete_model(ModelName=model_name)
        print(f"Deleted model: {model_name}")
    except Exception as exc:
        print(f"Model delete skipped or failed: {exc}")